# Lab 3: Evaluate retrieval failures before production

## Business problem

A successful demo question does not prove that retrieval is reliable. Policies can contain exact identifiers, duplicate wording, outdated versions, conflicting rules, and missing answers.

## Mission

Test the HR retrieval pipeline against known failure patterns, record the expected evidence, and calculate a simple retrieval score. This turns manual experimentation into the beginning of an LLMOps evaluation practice.


## Exercise 1: Create a controlled policy corpus

**Mission:** Build test documents that represent conditions found in enterprise policy collections.


In [ ]:
from dotenv import load_dotenv
from langchain_core.documents import Document
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_openai import OpenAIEmbeddings

load_dotenv()

documents = [
    Document(
        page_content="Employees enrolled in benefit plan HMO-204 pay a $35 specialist copay.",
        metadata={"source": "benefits_2026.md", "section": "HMO-204", "status": "current"},
    ),
    Document(
        page_content="Employees enrolled in benefit plan PPO-440 pay a $60 specialist copay.",
        metadata={"source": "benefits_2026.md", "section": "PPO-440", "status": "current"},
    ),
    Document(
        page_content="Full-time employees receive 12 weeks of paid parental leave.",
        metadata={"source": "leave_2024.md", "section": "Parental Leave", "status": "archived"},
    ),
    Document(
        page_content="Full-time employees receive 16 weeks of paid parental leave.",
        metadata={"source": "leave_2026.md", "section": "Parental Leave", "status": "current"},
    ),
    Document(
        page_content="Submit time off requests in the HR portal at least five business days in advance when possible.",
        metadata={"source": "time_off_2026.md", "section": "Requests", "status": "current"},
    ),
    Document(
        page_content="Managers review time off requests in the HR portal and respond within two business days.",
        metadata={"source": "manager_guide_2026.md", "section": "Approvals", "status": "current"},
    ),
]

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vector_store = InMemoryVectorStore.from_documents(documents, embedding=embeddings)

print("Indexed test documents:", len(documents))


## Exercise 2: Exact identifiers

**Mission:** Check whether semantic retrieval preserves an exact plan code.

**Risk:** Similar descriptions can rank above the document containing the exact identifier.


In [ ]:
question = "What is the specialist copay for HMO-204?"
results = vector_store.similarity_search(question, k=3)

for result in results:
    print(result.metadata)
    print(result.page_content)
    print()


### Inspect the result

`HMO-204` should rank first. If it does not, the system may need hybrid retrieval that combines keyword matching with vector similarity. Week 3 identifies this operational requirement without introducing agentic retrieval.


## Exercise 3: Conflicting policy versions

**Mission:** Observe what happens when current and archived policies contain different answers.


In [ ]:
question = "How many weeks of paid parental leave do full-time employees receive?"
results = vector_store.similarity_search(question, k=3)

for result in results:
    print(result.metadata)
    print(result.page_content)
    print()


### What this proves

Similarity alone does not know which policy is authoritative. Production retrieval needs lifecycle metadata and a rule that excludes archived content from employee answers.


## Exercise 4: Filter archived content

**Mission:** Build an index containing only approved current policies and rerun the same question.


In [ ]:
current_documents = []

for document in documents:
    if document.metadata["status"] == "current":
        current_documents.append(document)

current_store = InMemoryVectorStore.from_documents(current_documents, embedding=embeddings)
results = current_store.similarity_search(question, k=3)

for result in results:
    print(result.metadata)
    print(result.page_content)
    print()


## Exercise 5: Missing information

**Mission:** Prove that nearest does not mean sufficient.


In [ ]:
question = "Does the company reimburse home internet service?"
results = current_store.similarity_search_with_score(question, k=3)

for document, score in results:
    print("Distance:", round(score, 3))
    print(document.metadata)
    print(document.page_content)
    print()


### What this proves

A vector store still returns the nearest available documents even when none answers the question. The application needs an insufficient-evidence policy that is evaluated with real questions. A similarity threshold can help, but it must be calibrated rather than guessed.


## Exercise 6: Turn questions into a retrieval evaluation

**Mission:** Record expected sources and measure whether the correct evidence appears in the first result.


In [ ]:
test_cases = [
    {
        "question": "What is the specialist copay for HMO-204?",
        "expected_source": "benefits_2026.md",
        "expected_section": "HMO-204",
    },
    {
        "question": "How many weeks of paid parental leave do full-time employees receive?",
        "expected_source": "leave_2026.md",
        "expected_section": "Parental Leave",
    },
    {
        "question": "Where should an employee submit a time off request?",
        "expected_source": "time_off_2026.md",
        "expected_section": "Requests",
    },
]

passed = 0

for test in test_cases:
    result = current_store.similarity_search(test["question"], k=1)[0]
    source_matches = result.metadata["source"] == test["expected_source"]
    section_matches = result.metadata["section"] == test["expected_section"]
    test_passed = source_matches and section_matches

    if test_passed:
        passed = passed + 1

    print("Question:", test["question"])
    print("Expected:", test["expected_source"], "|", test["expected_section"])
    print("Retrieved:", result.metadata["source"], "|", result.metadata["section"])
    print("PASS" if test_passed else "FAIL")
    print()

score = passed / len(test_cases)
print("Retrieval accuracy:", round(score * 100), "%")


## Exercise 7: Experiment with one moving piece

Choose one controlled change:

- change the embedding model;
- change the chunk size in the Lab 1 pipeline;
- change `k`;
- include archived documents;
- add a new HR policy;
- rewrite one test question as an employee might actually ask it.

Run the same evaluation before and after the change. Record the configuration, score, failed questions, latency, and estimated embedding cost. An LLMOps engineer promotes a pipeline change based on evidence, not because one demo looked convincing.


## Week 3 checkpoint

You can now trace and test the complete retrieval path:

`source -> parse -> chunk -> metadata -> embed -> index -> retrieve -> augment -> generate -> evaluate`

The notebooks expose AI engineering decisions so an LLMOps engineer can reproduce them, monitor them, compare versions, and diagnose failures in production.
